In [10]:
import yt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import bisect
from scipy.spatial import cKDTree


In [19]:
# given plt file directory
# get the flame front point 
def read_get_flamex(d1):
    import yt
    import numpy as np
    import pandas as pd
    #load ds
    ds = yt.load(d1)
    ds.force_periodicity()
    # fields we will load per grid
    grad_keys = [
        ("boxlib", "Y(H2)_gradient_x"),
        ("boxlib", "Y(H2)_gradient_y"),
        ("boxlib", "Y(H2)_gradient_z"),
    ]
    Tkey = ("boxlib", "temp")
    
    # Collect chunks here
    chunks = []
    for g in ds.index.grids:
   
        if float(g.LeftEdge[0]) >= 0.07:
            continue
    
        # Pull minimal fields first (to build mask cheaply)
        x = g[("index", "x")].ndarray_view().ravel()
        T = g[Tkey].ndarray_view().ravel()
    
        mask = (T > 310.0) & (x < 0.07)
        if not np.any(mask):
            continue
    
        # Now pull only what you need (still per-grid, so small)
        y  = g[("index", "y")].ndarray_view().ravel()
        z  = g[("index", "z")].ndarray_view().ravel()
    
        data = {
            "x": x[mask],
            "y": y[mask],
            "z": z[mask],
            "T": T[mask],
        }
        chunks.append(pd.DataFrame(data))
    
    df_350 = pd.concat(chunks, ignore_index=True)
    #print(df_350['x'].min(),df_350['x'].max())
    x_cold_region = df_350['x'].min()-0.003

    return x_cold_region





# get flame region cut df
def get_flame_cut(d1,Tcut:float=315,x_cold_region:float=0):
    import yt
    import numpy as np
    import pandas as pd
    #load ds
    ds = yt.load(d1)
    ds.force_periodicity()
    # add the 
    ds.add_gradient_fields(("boxlib", "Y(H2)"))
    Tkey = ("boxlib", "temp")
    mass_fields = [f for f in ds.field_list if "Y(" in f[1]]
    # fields we will load per grid
    grad_keys = [
        ("boxlib", "Y(H2)_gradient_x"),
        ("boxlib", "Y(H2)_gradient_y"),
        ("boxlib", "Y(H2)_gradient_z"),
    ]
  

    # Collect chunks here
    chunks = []
    
    
    for g in ds.index.grids:
        # Quick reject: if this grid is entirely to the right of x=0.07, skip it
    
        # Pull minimal fields first (to build mask cheaply)
        x = g[("index", "x")].ndarray_view().ravel()
        T = g[Tkey].ndarray_view().ravel()
    
    
        """
        Just cut the region at the boundary of isothermal wall x=0.07
        If need to find points of flame at isothermal wall, make sure set the value larger than 0.07 (e.g. x< 0.075)
        """
        mask = (x>=x_cold_region) & (x < 0.075)
        if not np.any(mask):
            continue
    
        # Now pull only what you need (still per-grid, so small)
        y  = g[("index", "y")].ndarray_view().ravel()
        z  = g[("index", "z")].ndarray_view().ravel()
        dx = g[("index", "dx")].ndarray_view().ravel()
    
        gh2x = g[grad_keys[0]].ndarray_view().ravel()
        gh2y = g[grad_keys[1]].ndarray_view().ravel()
        gh2z = g[grad_keys[2]].ndarray_view().ravel()
        
        gh2mag = np.sqrt(gh2x*gh2x + gh2y*gh2y + gh2z*gh2z)
        
        heat_release = g[('boxlib', 'HeatRelease')].ndarray_view().ravel()
        
        data = {
            "x": x[mask],
            "y": y[mask],
            "z": z[mask],
            "T": T[mask],
            "gridsize": dx[mask],
            "gh2x": gh2x[mask],
            "gh2y": gh2y[mask],
            "gh2z": gh2z[mask],
            "gh2mag": gh2mag[mask],
            "Q":heat_release[mask],
        }
            # mass fractions (only masked cells)
        for f in mass_fields:
            data[f[1]] = g[f].ndarray_view().ravel()[mask]
    
        chunks.append(pd.DataFrame(data))




    if len(chunks) == 0:
        raise RuntimeError("No cells found for (T>350) & (x<0.07).")

    df_cut = pd.concat(chunks, ignore_index=True)
    df_cut['global_index'] =df_cut.index
    
    min_idx = df_cut[df_cut['T']>Tcut]['x'].idxmin()
    #print(min_idx)
    #return df_cut, [df_cut['x'].iloc[min_idx], df_cut['y'].iloc[min_idx], df_cut['z'].iloc[min_idx] ]
    return df_cut, min_idx


In [31]:
def find_pathline(df, x0,y0,z0,cutoff_value,max_length,curr_dx):
    pathline = []
    pathline_dict =[]
    find_next_point_cubic(df,x0,y0,z0,cutoff_value,max_length,curr_dx,pathline,pathline_dict)
    
    return pathline,pathline_dict




def find_next_point_cubic(df,x0,y0,z0,cutoff_value,max_length,curr_dx,pathline,pathline_dict):
    # if the line is long enough, then return
    if len(pathline)>=2:
        pt0 = pathline[0]
        ptlast = pathline[-1]
        distance = ((pt0[0] - ptlast[0] )**2  + (pt0[1] - ptlast[1]  )**2 + (pt0[2] - ptlast[2]  )**2 )**0.5
        if distance >= cutoff_value:
            return 
    if len(pathline_dict) >= max_length:
            return 

    temp_weight_dict = get_weight_w_point(df,x0,y0,z0,curr_dx)
    #print(temp_weight_dict)

    # make sure the indexes of pathline and pathline dict are aligned
    pathline.append([x0,y0,z0])
    pathline_dict.append(temp_weight_dict)


    # that's the c_i/ |c| part 
    x_dirc, y_dirc, z_dirc = 0,0,0
    next_dx  =  0
    for k,v in temp_weight_dict.items():
        x_dirc += (df['gh2x'].iloc[k] * v)
        y_dirc += (df['gh2y'].iloc[k] * v)
        z_dirc += (df['gh2z'].iloc[k] * v)
        #next_dx = max(next_dx, df['gridsize'].iloc[k])
    next_dx = curr_dx
    norm_dist = (x_dirc**2+ y_dirc**2+z_dirc**2)*0.5
    norm_dist_sq = (x_dirc**2+ y_dirc**2+z_dirc**2)
    x_next = x0  - x_dirc/norm_dist_sq * (next_dx*0.15*norm_dist)
    y_next = y0  - y_dirc/norm_dist_sq * (next_dx*0.15*norm_dist)
    z_next = z0  - z_dirc/norm_dist_sq * (next_dx*0.15*norm_dist)
    if z_next <=0:
        return
    #print(f'The next point is x: {x_next}, y: {y_next} , z: {z_next}')

    find_next_point_cubic(df,x_next,y_next,z_next,cutoff_value,max_length,next_dx,pathline,pathline_dict)
        
    return 




"""
need to have 2 separate method to extract points

"""

# supposed to return a directionary/list with 8 pts (index)

"""
def find_pts(df,x0,y0,z0,curr_dx):
    pts_good = []
    coord_dict = {}
    
        
    xbl, xbr = x0 - 4*curr_dx, x0 + 4*curr_dx
    ybl, ybr = y0 - 4*curr_dx, y0 + 4*curr_dx
    zbl, zbr = z0 - 4*curr_dx, z0 + 4*curr_dx

    
     
    if (xbl>0.07) or (xbr>0.07):
        xbl -= 2*curr_dx
        xbr += 2*curr_dx
        ybl -= 2*curr_dx
        ybr += 2*curr_dx
        zbl -= 2*curr_dx
        zbr += 2*curr_dx
    
    # the 01 indiation of cubic
    coord_idx = [[-1,-1,-1], [-1,-1,1], [-1,1,-1],[-1,1,1],
                [1,-1,-1], [1,-1,1], [1,1,-1],[1,1,1], ]

    print(f'for {x0}, {y0}, {z0}.')
    for ec in coord_idx:
        x_edge = xbr if ec[0] ==1 else xbl
        y_edge = ybr if ec[1] ==1 else ybl
        z_edge = zbr if ec[2] ==1 else zbl
        print(f'xedge is {x_edge}')

        subxl, subxr = min(x_edge,x0), max(x_edge,x0)
        subyl, subyr = min(y_edge,y0), max(y_edge,y0)
        subzl, subzr = min(z_edge,z0), max(z_edge,z0)

        print(f'Find from x : {subxl} to {subxr}.')
        print(f'Find from y : {subyl} to {subyr}.')
        print(f'Find from z : {subzl} to {subzr}.')

        temp_cut = df[ (df['x'] > subxl) &  (df['x'] <= subxr) & (df['y'] > subyl) &  (df['y'] <= subyr) & (df['z'] > subzl) &  (df['z'] <= subzr)]
        df_cut = temp_cut.copy()

        points = df_cut[['x','y','z']].to_numpy()
        print(points)
        tree = cKDTree(points)
        
        query_point = np.array([x0, y0, z0])
        dist, i_local = tree.query(query_point)
        print(i_local)
        row = df.iloc[int(i_local)] 
        row_cut = df.iloc[int(i_local)]

        # row inside df_cut (guaranteed in-range)
        idx_global = df.index[int(i_local)]  # original df index label
        print(f"find the near point of {df[['x','y','z']].iloc[idx_global]}.  ")
        pts_good.append(idx_global)
        coord_dict[tuple(ec)] = idx_global
        
    

    return pts_good, coord_dict
"""
def find_pts(df,x0,y0,z0,curr_dx):
    pts_good = []
    coord_dict = {}
    
    
    xbl, xbr = x0 - 4*curr_dx, x0 + 4*curr_dx
    ybl, ybr = y0 - 4*curr_dx, y0 + 4*curr_dx
    zbl, zbr = z0 - 4*curr_dx, z0 + 4*curr_dx
    #print(f' trying to find points for {x0},{y0},{z0}')
    # the 01 indiation of cubic
    coord_idx = [[-1,-1,-1], [-1,-1,1], [-1,1,-1],[-1,1,1],
                [1,-1,-1], [1,-1,1], [1,1,-1],[1,1,1], ]


    for ec in coord_idx:
        x_edge = xbr if ec[0] ==1 else xbl
        y_edge = ybr if ec[1] ==1 else ybl
        z_edge = zbr if ec[2] ==1 else zbl

        subxl, subxr = min(x_edge,x0), max(x_edge,x0)
        subyl, subyr = min(y_edge,y0), max(y_edge,y0)
        subzl, subzr = min(z_edge,z0), max(z_edge,z0)


        if subzl <0:
            subzl += curr_dx/2
            subzr += curr_dx/2

        if (subxl >0.07 ) or (subxr >0.07):
            subxl += 3*curr_dx
            subxr += 3*curr_dx

        
        temp_cut = df[ (df['x'] > subxl) &  (df['x'] <= subxr) & (df['y'] > subyl) &  (df['y'] <= subyr) & (df['z'] > subzl) &  (df['z'] <= subzr)]


        
        df_cut = temp_cut.copy()
        #
        if (df_cut.shape[0]==0):
            print(f'no point can be found for x:{subxl }:{subxr }, y: {subyl }:{subyr }, z:{subzl }:{subzr }')
        points = df_cut[['x','y','z']].to_numpy()
        tree = cKDTree(points)
        
        #print(f'subxl: {subxl}, subxr: {subxr}')
        #print(f'subyl: {subyl}, subyr: {subyr}')
        #print(f'subzl: {subzl}, subzr: {subzr}')
        #print(df_cut['x'].min(), df_cut['x'].max())
        query_point = np.array([x0, y0, z0])
        # index is the index of the original df
        # and this index need to be stored and used
        #dist, idx = tree.query(query_point)
        #print(f"for {ec} is nearest index is {idx} with distance {dist} x:{df_out['x'].loc[idx]}, y:{df_out['y'].loc[idx]}, z:{df_out['z'].loc[idx]}.")
        dist, i_local = tree.query(query_point)
        if(subxl>subxr):
            print('something wrong on x !')
            break

        if(subyl>subyr):
            print('something wrong on y !')
            break

        if(subzl>subzr):
            print('something wrong on z!')
            break
        
        idx_global = df_cut['global_index'].iloc[int(i_local)]
        
        #print(f'local index is {i_local} and global index is {idx_global}.')
        #print(f"xyz: {df[['x','y','z']].iloc[idx_global]}")
        
        pts_good.append(idx_global)
        coord_dict[tuple(ec)] = idx_global
        


    return pts_good, coord_dict








In [14]:
#print(pts1[0],pts1[1],pts1[2])
#pathline,pathdict = find_pathline(df1,pts1[0],pts1[1],pts1[2],0.0035,200,34.1e-6  )

In [26]:

def get_weight_w_point(df0,x0,y0,z0,curr_dx):

    df =df0.copy()
    temp_ls, _ = find_pts(df,x0,y0,z0,curr_dx)
    
    temp_dist = []
    subdf = df[['x', 'y', 'z']].loc[temp_ls]
    #print(f'the length of subdf is {subdf.shape[0]}')
    minv = 1e-12
    currmin = 100
    for i in range(subdf.shape[0]):
        tx = subdf['x'].iloc[i]
        ty = subdf['y'].iloc[i]
        tz = subdf['z'].iloc[i]
        minxyz_pre = [(tx - x0  )**2, (ty - y0  )**2 , (tz- z0  )**2 ] 
        
        minxyz = [ x for x in minxyz_pre if x>0]
        minxyz.append(currmin)
        currmin = min(minxyz)
        #print(f'for current {tx}, {ty}, {tz}, the diff is {minxyz}')

    minv = min(1e-4, currmin/1000)

    

    #print(minv)
    
    for i in range(subdf.shape[0]):
        x_diff = subdf['x'].iloc[i] - x0 if (subdf['x'].iloc[i] - x0)!=0 else minv
        y_diff = subdf['y'].iloc[i] - y0 if (subdf['y'].iloc[i] - y0)!=0 else minv
        z_diff = subdf['z'].iloc[i] - z0 if (subdf['z'].iloc[i] - z0)!=0 else minv

        
        temp_dist.append( 1/( (x_diff)**2 +   (y_diff)**2 +( z_diff)**2) )

    sum_weight = sum(temp_dist)
    weight_dict = {}
    check_ttl=0

    # find way to 
    for i in range(subdf.shape[0]):
        if subdf.index[i] not in weight_dict:
            weight_dict[subdf.index[i]] = temp_dist[i] /  sum_weight
        else:
            weight_dict[subdf.index[i]] += temp_dist[i] /  sum_weight

    #print(check_ttl)
    return weight_dict


def get_T_Mass_from_list(df,pathline,pathline_dict):
    index_col = ['x', 'y', 'z']
    header_col = [ec for ec in df.columns if 'Y('  in ec ]
    header_col.append('T')
    
    header_col.extend( ['gh2x','gh2y', 'gh2z','Q'])
    heat_release = ["Q"]
    full_col = index_col+ header_col 
    #final_df = pd.DataFrame(columns=full_col)
    # this list combined 2 items, [index, value]
    # then sort by index. 
    final_list = []
    for i in range(len(pathline_dict)):
        temp_dict = pathline_dict[i]
        temp_target_coord = pathline[i]
        
        temp_intp_list = []
        temp_intp_list.extend(pathline[i])
        temp_intp_weight = []
        #print(temp_dict)
        w=pd.Series(temp_dict, name="weight")
        temo_subdf = df[header_col].loc[w.index].copy()
        temo_subdf["weight"] = w
        #print(temo_subdf)
        #num_cols = df_selected.select_dtypes(include="number").columns
        df_weighted = temo_subdf[header_col].multiply(temo_subdf["weight"], axis=0)
        
        for i in range(len(header_col)):
            #print(df_weighted[header_col[i]])
            temp_intp_list.append(sum(df_weighted[header_col[i]]))
        final_list.append(temp_intp_list)
    
        
    return pd.DataFrame(final_list,columns = full_col)
            
            


In [17]:

#pt1 = read_get_flamex("/Users/potato/Downloads/plt59250",315)
#df1,min_idx = get_flame_cut("/Users/potato/Downloads/plt59250",315,pt1)



In [27]:
d1 = "/Users/potato/Downloads/plt59250"
plt_name = d1.split('/')
pt1 = read_get_flamex("/Users/potato/Downloads/plt59250")

df1,min_idx = get_flame_cut("/Users/potato/Downloads/plt59250",303,pt1)
pts1 = df1[['x','y','z']].iloc[min_idx]


yt : [INFO     ] 2026-03-04 10:33:48,146 Parameters: current_time              = 0.1696800000000185
yt : [INFO     ] 2026-03-04 10:33:48,146 Parameters: domain_dimensions         = [384  64  64]
yt : [INFO     ] 2026-03-04 10:33:48,147 Parameters: domain_left_edge          = [0. 0. 0.]
yt : [INFO     ] 2026-03-04 10:33:48,148 Parameters: domain_right_edge         = [0.105  0.0175 0.0175]
yt : [INFO     ] 2026-03-04 10:33:51,654 Parameters: current_time              = 0.1696800000000185
yt : [INFO     ] 2026-03-04 10:33:51,656 Parameters: domain_dimensions         = [384  64  64]
yt : [INFO     ] 2026-03-04 10:33:51,656 Parameters: domain_left_edge          = [0. 0. 0.]
yt : [INFO     ] 2026-03-04 10:33:51,657 Parameters: domain_right_edge         = [0.105  0.0175 0.0175]


In [ ]:


pathline,pathdict = find_pathline(df1,pts1[0],pts1[1],pts1[2],0.0035,300,34.1e-6  )
df_p1 = get_T_Mass_from_list(df1,pathline,pathdict)
df_p1['c'] = 1- df_p1['Y(H2)'] /0.01304 
df_p1

In [28]:
df1

,x,y,z,T,gridsize,gh2x,gh2y,gh2z,gh2mag,Q,Y(H),Y(H2),Y(H2O),Y(H2O2),Y(HO2),Y(N2),Y(O),Y(O2),Y(OH),global_index
0,0.062754,0.000137,0.000137,298.000234,0.000273,-1.853451e-11,-6.572520e-12,-1.006804e-10,1.025829e-10,-3.781801e-03,-1.749652e-17,0.013040,-4.607740e-20,3.252954e-21,1.384896e-15,0.756999,2.002840e-25,0.229961,-1.462288e-22,0
1,0.062754,0.000137,0.000410,298.000214,0.000273,-1.512035e-11,-9.876544e-13,6.391687e-11,6.568841e-11,9.038261e-03,3.715790e-17,0.013040,-4.746962e-20,3.038055e-21,-8.429508e-15,0.756999,-1.288393e-24,0.229961,-9.427782e-23,1
2,0.062754,0.000137,0.000684,298.000196,0.000273,-1.512390e-11,-1.009681e-11,5.683631e-11,5.967449e-11,-3.209867e-02,6.360492e-17,0.013040,-4.522897e-20,2.823656e-21,-6.276937e-16,0.756999,-1.384084e-25,0.229961,-1.057287e-22,2
3,0.062754,0.000137,0.000957,298.000142,0.000273,-1.462297e-11,-2.144418e-11,3.708323e-11,4.526422e-11,1.163654e-02,1.820824e-17,0.013040,-3.594272e-20,2.122796e-21,-1.835485e-15,0.756999,-2.368759e-25,0.229961,-1.147436e-22,3
4,0.062754,0.000137,0.001230,298.000106,0.000273,-1.655565e-12,-8.540724e-12,1.084643e-11,1.390432e-11,7.363535e-03,-2.797661e-17,0.013040,-2.958404e-20,1.654140e-21,-6.081219e-16,0.756999,-9.005388e-26,0.229961,-8.251573e-23,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3885051,0.074973,0.017483,0.000120,1570.051826,0.000034,-1.029312e-02,1.255721e-02,2.172551e-02,2.712250e-02,2.389604e+07,5.988214e-06,0.000048,1.446789e-01,2.466783e-06,7.139161e-06,0.758663,2.804310e-04,0.094765,1.549364e-03,3885051
3885052,0.074973,0.017483,0.000154,1578.392662,0.000034,-1.050570e-02,1.296481e-02,2.677011e-02,3.154512e-02,2.377033e+07,6.100070e-06,0.000048,1.446028e-01,2.403197e-06,7.030947e-06,0.758664,2.863393e-04,0.094797,1.585845e-03,3885052
3885053,0.074973,0.017483,0.000188,1586.391040,0.000034,-1.075910e-02,1.346909e-02,3.157473e-02,3.597413e-02,2.389966e+07,6.252730e-06,0.000049,1.445020e-01,2.355131e-06,6.953943e-06,0.758660,2.936079e-04,0.094853,1.626023e-03,3885053
3885054,0.074973,0.017483,0.000222,1594.027078,0.000034,-1.105124e-02,1.406943e-02,3.614880e-02,4.033379e-02,2.422913e+07,6.444695e-06,0.000051,1.443743e-01,2.319974e-06,6.906390e-06,0.758650,3.021645e-04,0.094937,1.669235e-03,3885054


In [9]:


pathline,pathdict = find_pathline(df1,df1['x'].iloc[min_idx],df1['y'].iloc[min_idx],df1['z'].iloc[min_idx],0.0035,250,34.1e-6  )
df_p1 = get_T_Mass_from_list(df1,pathline,pathdict)
df_wholefront = df_p1.loc[df_p1.groupby('y')['x'].idxmin()].reset_index(drop=True)
df_wholefront.to_csv(f'{plt_name}_flame_front_pts.csv', index=False)


In [97]:
pts1 = df1[['x','y','z']].iloc[min_idx]
pathline,pathdict = find_pathline(df1,pts1[0],pts1[1],pts1[2],0.0035,250,34.1e-6  )
df_p1 = get_T_Mass_from_list(df1,pathline,pathdict)
df_p1['c'] = 1- df_p1['Y(H2)'] /0.01304 
df_p1

,x,y,z,Y(H),Y(H2),Y(H2O),Y(H2O2),Y(HO2),Y(N2),Y(O),Y(O2),Y(OH),T,gh2x,gh2y,gh2z,c
0,0.065505,0.006853,0.000188,3.533262e-15,0.012673,0.007429,0.000041,0.000003,0.756925,6.939839e-11,0.222929,1.066016e-11,303.056633,-2.362505,-0.251473,0.415128,0.028179
1,0.065515,0.006854,0.000186,3.778822e-15,0.012649,0.007616,0.000042,0.000004,0.756917,7.776061e-11,0.222772,1.276715e-11,303.528816,-2.504155,-0.253623,0.364083,0.029990
2,0.065526,0.006855,0.000185,4.466097e-15,0.012612,0.007917,0.000044,0.000004,0.756905,9.259578e-11,0.222518,1.648307e-11,304.418871,-2.727557,-0.277701,0.352433,0.032828
3,0.065538,0.006856,0.000183,5.092139e-15,0.012589,0.008110,0.000046,0.000004,0.756898,1.036016e-10,0.222352,1.923623e-11,305.127552,-2.862189,-0.314234,0.418581,0.034548
4,0.065550,0.006858,0.000181,6.833728e-15,0.012550,0.008391,0.000048,0.000005,0.756885,1.215471e-10,0.222121,2.490665e-11,306.159816,-3.092442,-0.334652,0.377563,0.037614
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
245,0.067092,0.007181,0.000188,4.705179e-05,0.000282,0.159895,0.000014,0.000043,0.759343,5.935671e-04,0.078572,1.209584e-03,1238.539068,-0.298295,-0.072900,-0.017244,0.978358
246,0.067093,0.007181,0.000188,5.119773e-05,0.000282,0.159735,0.000013,0.000040,0.759487,6.641941e-04,0.078356,1.371046e-03,1272.181588,-0.297520,-0.074770,0.045658,0.978357
247,0.067094,0.007181,0.000188,4.689898e-05,0.000282,0.159884,0.000014,0.000043,0.759340,5.922520e-04,0.078590,1.207784e-03,1238.402994,-0.296933,-0.072335,-0.016929,0.978405
248,0.067095,0.007182,0.000188,4.653939e-05,0.000280,0.159884,0.000014,0.000043,0.759337,5.889894e-04,0.078603,1.203206e-03,1237.721859,-0.293011,-0.072485,-0.015515,0.978524


In [11]:
df_whole_front = df1[(df1['x']<=0.07) & (df1['z'] > 2*34e-6) & (df1['y']<=0.015) & (df1['y'] >=0.002) & (df1['T']>=305)& (df1['x']<= df1['x'].iloc[min_idx]+0.002 ) ]
df_sub = df_whole_front.loc[df_whole_front.groupby('y')['x'].idxmin()].reset_index(drop=True)
df_fb = df_sub[df_sub['x']< df1['x'].iloc[min_idx]+0.002 ] 
df_stable = df_sub[df_stable['x']> 0.069]

In [12]:
xs2 = df_sub['x'].iloc[1]
ys2 = df_sub['y'].iloc[1]
zs2 = df_sub['z'].iloc[1]
print(f'try this {xs2}, {ys2},{zs2}.')

try this 0.06967529296874998, 0.0020336914062500005,8.544921875000001e-05.


In [13]:
pl2,pd2 = find_pathline(df1,  xs2  ,  ys2 ,  zs2  ,0.0035,250,68.1e-6  )
df_p2 = get_T_Mass_from_list(df1,pl2,pd2)

In [ ]:
sf_oh=[]
fb_oh=[]

sf_o2=[]
fb_o2=[]

sf_o=[]
fb_o=[]

sf_ho2=[]
fb_ho2=[]

sf_h2o2=[]
fb_h2o2=[]

sf_h2o=[]
fb_h2o=[]

sf_h2=[]
fb_h2=[]

sf_cline= []
sf_t = []
fb_cline = []
fb_t = []


In [ ]:
x_cutoff = df1['x'].iloc[min_idx]+0.002
for i in range(df_sub.shape[0]):
    temp_xs  = df_sub['x'].iloc[i]
    temp_ys  = df_sub['y'].iloc[i]
    temp_zs  = df_sub['z'].iloc[i]
    try:
        tpl2,tpd2 = find_pathline(df1,  temp_xs  ,  temp_ys ,  temp_zs  ,0.0035,250,68.1e-6  )
        temp_df2 = get_T_Mass_from_list(df1,tpl2,tpd2) 
        temp_cline = [1- temp_df2['Y(H2)'] /0.01304 ]
        temp_oh = [temp_df2['Y(OH)']]
        temp_o2 = [temp_df2['Y(O2)']]
        temp_o = [temp_df2['Y(o)']]
        temp_ho2 = [temp_df2['Y(HO2)']]
        temp_h2o2 = [temp_df2['Y(H2O2)']]
        temp_h2o = [temp_df2['Y(H2O)']]    
        temp_t2 = [temp_df2['T'] ]
        if temp_xs<=x_cutoff:
            fb_oh.append(temp_oh)
            fb_o2.append(temp_o2)
            fb_o.append(temp_o)
            fb_ho2.append(temp_ho2)
            fb_h2o2.append(temp_h2o2)
            fb_h2o.append(temp_h2o)
            #fb_h2.append(temp_h2)
            fb_cline.append(temp_cline)
        else:
            sf_oh.append(temp_oh)
            sf_o2.append(temp_o2)
            sf_o.append(temp_o)
            sf_ho2.append(temp_ho2)
            sf_h2o2.append(temp_h2o2)
            sf_h2o.append(temp_h2o)
            sf_cline.append(temp_cline)
        
    
        print(f'The point of {temp_xs}, {temp_ys},{temp_zs} find c line.')
    except:
        print(f'The point of {temp_xs}, {temp_ys},{temp_zs} cant get progress variable line')


The point of 0.06969238281249998, 0.0020166015625000004,0.00010253906250000001 find c line.
The point of 0.06967529296874998, 0.0020336914062500005,8.544921875000001e-05 find c line.
The point of 0.06986328124999999, 0.00205078125,0.00013671875 find c line.
The point of 0.06967529296874998, 0.00206787109375,8.544921875000001e-05 find c line.
The point of 0.06969238281249998, 0.0020849609375000002,0.00010253906250000001 find c line.
The point of 0.06967529296874998, 0.0021020507812500003,8.544921875000001e-05 find c line.
The point of 0.06979492187499999, 0.0021191406250000004,6.8359375e-05 find c line.
The point of 0.06967529296874998, 0.00213623046875,8.544921875000001e-05 find c line.
The point of 0.06969238281249998, 0.0021533203125,0.00010253906250000001 find c line.
The point of 0.06967529296874998, 0.00217041015625,8.544921875000001e-05 find c line.
The point of 0.06967529296874998, 0.0022045898437500003,8.544921875000001e-05 find c line.
The point of 0.06969238281249998, 0.00222

In [71]:

df1['grid_index'] = round(df1['y']/ (34.1e-6),3)
df1f = df1[df1['T']>=305].reset_index(drop=True).copy()
y_unique = set(df1f['grid_index'])
y_index =[]
for i in y_unique:
    temp_df = df1f[df1f['grid_index']==i].sort_values(by='x',ascending=True).reset_index(drop=True)
    y_index.append(temp_df['global_index'].iloc[0])

df_r1 = df1.iloc[y_index].sort_values(by='y').reset_index(drop=True)


,x,y,z,T,gridsize,gh2x,gh2y,gh2z,gh2mag,Y(H),Y(H2),Y(H2O),Y(H2O2),Y(HO2),Y(N2),Y(O),Y(O2),Y(OH),global_index,grid_index
0,0.069436,0.000017,0.000188,305.241612,0.000034,-6.843981,0.946491,2.573065,7.372692,4.370926e-14,0.011366,0.003830,0.000026,0.000006,0.757240,5.304104e-10,0.227533,1.346357e-10,2488581,0.501
1,0.069487,0.000034,0.000103,306.872330,0.000068,-8.892011,1.009659,1.820657,9.132473,8.525623e-14,0.010785,0.006507,0.000043,0.000009,0.756973,6.917101e-10,0.225682,2.745010e-10,708097,1.002
2,0.069470,0.000051,0.000120,306.073019,0.000034,-8.166163,1.152815,2.085580,8.506753,6.096894e-14,0.010983,0.005748,0.000038,0.000008,0.757043,5.834829e-10,0.226180,1.981652e-10,2488843,1.504
3,0.069521,0.000068,0.000068,306.879455,0.000137,-10.171066,1.282368,-5.736508,11.747449,1.880191e-13,0.010447,0.008880,0.000057,0.000013,0.756686,1.059550e-09,0.223917,6.219108e-10,326656,2.005
4,0.069470,0.000085,0.000120,305.617664,0.000034,-8.015091,1.437714,2.217490,8.439548,5.971032e-14,0.011028,0.005895,0.000039,0.000008,0.757009,5.678285e-10,0.226021,1.922919e-10,2488851,2.506
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
955,0.069436,0.017415,0.000188,305.309731,0.000034,-7.123558,0.048165,2.326769,7.494081,4.674419e-14,0.011317,0.003154,0.000020,0.000005,0.757381,5.874644e-10,0.228122,1.418238e-10,3806189,510.691
956,0.069521,0.017432,0.000068,307.373738,0.000137,-10.472517,0.247388,-6.229535,12.187777,2.083080e-13,0.010343,0.007757,0.000048,0.000012,0.756876,1.130630e-09,0.224963,6.543818e-10,519408,511.192
957,0.069436,0.017449,0.000188,305.453086,0.000034,-7.070185,0.316151,2.379400,7.466526,4.665346e-14,0.011323,0.003396,0.000022,0.000005,0.757334,5.721597e-10,0.227920,1.415248e-10,3806197,511.693
958,0.069487,0.017466,0.000103,307.111017,0.000068,-9.081046,0.429771,1.666985,9.242777,8.951243e-14,0.010737,0.005922,0.000038,0.000009,0.757081,7.258503e-10,0.226214,2.836947e-10,2294689,512.194


In [84]:
l1= df_r1[ (df_r1['z']<=(3*34.1e-6)) &(df_r1['x'] <=0.068)]

In [86]:
l1

,x,y,z,T,gridsize,gh2x,gh2y,gh2z,gh2mag,Y(H),Y(H2),Y(H2O),Y(H2O2),Y(HO2),Y(N2),Y(O),Y(O2),Y(OH),global_index,grid_index
318,0.067607,0.005811,0.000068,305.739380,0.000137,-2.069170,-9.227636,-2.719424,9.840020,-1.817306e-13,0.011351,0.019184,0.000102,0.000013,0.756568,4.923676e-10,0.212783,2.692024e-10,240288,170.397
333,0.066514,0.006084,0.000068,307.100629,0.000137,-3.875930,-9.982537,-2.641263,11.029513,-4.163340e-12,0.011694,0.010414,0.000055,0.000011,0.757173,2.439720e-09,0.220653,2.753398e-09,238272,178.416
348,0.065967,0.006357,0.000068,306.937284,0.000137,-4.804887,-5.239474,-2.498651,7.535402,-3.840976e-13,0.012030,0.010280,0.000054,0.000008,0.757174,6.116170e-10,0.220454,3.251816e-10,237280,186.435
356,0.065830,0.006494,0.000068,308.046253,0.000137,-5.078753,-3.650436,-2.584691,6.767573,-1.325346e-13,0.012065,0.011014,0.000059,0.000008,0.757064,3.873578e-10,0.219789,1.666100e-10,237040,190.444
363,0.065693,0.006631,0.000068,305.043247,0.000137,-4.271898,-1.842025,-1.978487,5.055352,-3.017878e-15,0.012300,0.010136,0.000054,0.000006,0.757001,1.617572e-10,0.220503,4.560923e-11,242688,194.453
393,0.065693,0.007178,0.000068,308.351554,0.000137,-6.360335,1.221409,-2.760387,7.040272,-1.596599e-13,0.012044,0.006753,0.000041,0.000008,0.757030,7.116216e-10,0.224124,3.228706e-10,242752,210.491
423,0.065967,0.007725,0.000068,308.134089,0.000137,-7.658406,5.058675,-3.902259,9.973415,-1.011303e-11,0.011882,0.001798,0.000012,0.000008,0.757477,5.071886e-09,0.228823,6.763410e-09,243328,226.528
461,0.066787,0.008408,0.000068,306.505181,0.000137,-4.745799,9.105783,-4.043148,11.035621,-4.166402e-11,0.011808,0.003830,0.000019,0.000008,0.757794,6.703644e-09,0.226540,1.045433e-08,244944,246.575
468,0.067061,0.008545,0.000068,305.363456,0.000137,-4.018711,8.857244,-3.682768,10.400173,-1.969113e-11,0.011796,0.005590,0.000025,0.000007,0.757934,3.824345e-09,0.224647,3.681235e-09,245472,250.584


In [ ]:
sf_oh=[]
fb_oh=[]

sf_o2=[]
fb_o2=[]

sf_o=[]
fb_o=[]

sf_ho2=[]
fb_ho2=[]

sf_h2o2=[]
fb_h2o2=[]

sf_h2o=[]
fb_h2o=[]

sf_h2=[]
fb_h2=[]

sf_cline= []
sf_t = []
fb_cline = []
fb_t = []

x_cutoff = df1['x'].iloc[min_idx]+0.002
for i in range(df_sub.shape[0]):
    temp_xs  = df_sub['x'].iloc[i]
    temp_ys  = df_sub['y'].iloc[i]
    temp_zs  = df_sub['z'].iloc[i]
    try:
        tpl2,tpd2 = find_pathline(df1,  temp_xs  ,  temp_ys ,  temp_zs  ,0.0035,250,68.1e-6  )
        temp_df2 = get_T_Mass_from_list(df1,tpl2,tpd2) 
        temp_cline = [1- temp_df2['Y(H2)'] /0.01304 ]
        temp_oh = [temp_df2['Y(OH)']]
        temp_o2 = [temp_df2['Y(O2)']]
        temp_o = [temp_df2['Y(o)']]
        temp_ho2 = [temp_df2['Y(HO2)']]
        temp_h2o2 = [temp_df2['Y(H2O2)']]
        temp_h2o = [temp_df2['Y(H2O)']]    
        temp_t2 = [temp_df2['T'] ]
        if temp_xs<=x_cutoff:
            fb_oh.append(temp_oh)
            fb_o2.append(temp_o2)
            fb_o.append(temp_o)
            fb_ho2.append(temp_ho2)
            fb_h2o2.append(temp_h2o2)
            fb_h2o.append(temp_h2o)
            #fb_h2.append(temp_h2)
            fb_cline.append(temp_cline)
        else:
            sf_oh.append(temp_oh)
            sf_o2.append(temp_o2)
            sf_o.append(temp_o)
            sf_ho2.append(temp_ho2)
            sf_h2o2.append(temp_h2o2)
            sf_h2o.append(temp_h2o)
            sf_cline.append(temp_cline)
        
    
        print(f'The point of {temp_xs}, {temp_ys},{temp_zs} find c line.')
    except:
        print(f'The point of {temp_xs}, {temp_ys},{temp_zs} cant get progress variable line')